# I-Z Spectroscopy Comparison
Comparing current-distance curves for Gold, UHV cleaved HOPG, and contaminated HOPG

In [1]:
import os
import numpy as np
import pandas as pd
import holoviews as hv
import hvplot.pandas
import rhkpy
import requests

hv.extension('bokeh')

In [3]:
print(f'numpy version: {np.__version__}')
print(f'holoviews version: {hv.__version__}')
print(f'rhkpy version: {rhkpy.__version__}')

numpy version: 2.3.1
holoviews version: 1.21.0
rhkpy version: 1.3.9


In [2]:
# Download source files from Zenodo if missing
ZENODO_FILES = {
    'iz_gold.sm4': 'https://zenodo.org/records/17469441/files/I-Z_gold_9K_2019_12_18_13_49_32_752.sm4?download=1',
    'iz_uhv.sm4': 'https://zenodo.org/records/17469441/files/I-Z_hopg_9K_2021_04_13_14_13_48_095.sm4?download=1',
    'iz_contaminated.sm4': 'https://zenodo.org/records/17469441/files/Stripes-9K-HOPG-SPI2-3_2021_09_09_05_50_41_043.sm4?download=1',
}

for local, url in ZENODO_FILES.items():
    if url is None or os.path.exists(local):
        continue
    print(f'Downloading {local}...')
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    with open(local, 'wb') as fh:
        fh.write(response.content)
    print(f'✓ {local} downloaded')

✓ iz_gold.sm4 downloaded
✓ iz_uhv.sm4 downloaded
✓ iz_contaminated.sm4 downloaded


In [3]:
# Load and process Gold data
gold_data = rhkpy.load_spym('iz_gold.sm4')
arr = gold_data.Current.to_numpy()

I_g = np.array([arr[i][4] * 1e12 for i in range(26)])
z_g = gold_data.Current.Current_x.to_numpy() * 1e10
I_g = I_g / max(I_g)  # normalize

gold_df = pd.DataFrame({'z': z_g, 'I': I_g})

In [4]:
# Load and process UHV cleaved HOPG data
uhv_data = rhkpy.rhkdata('iz_uhv.sm4').spectra

z_uhv = abs(uhv_data.variables['z'].to_numpy()) * 10
I_uhv = abs(uhv_data.variables['current'][:, 5, 1].to_numpy())
I_uhv = (I_uhv - 5) / max(I_uhv)

uhv_df = pd.DataFrame({'z': z_uhv, 'I': I_uhv})

In [5]:
iz_map = rhkpy.rhkdata('iz_contaminated.sm4')

In [22]:
# Load and process contaminated HOPG data
iz_map = rhkpy.rhkdata('iz_contaminated.sm4')
izmap_selected = iz_map.spectra.current.sel(
	zscandir='up',
	repetitions=1
).sel(
    specpos_x=-174.41,
	specpos_y=-128.12,
    method='nearest'
)

z_alkane = iz_map.spectra.variables['z'].to_numpy() * 10
I_alkane = abs(izmap_selected).to_numpy()
I_alkane = np.flip(I_alkane / max(I_alkane))
z_alkane = np.flip(z_alkane)

alkane_df = pd.DataFrame({'z': z_alkane, 'I': I_alkane})

In [23]:
# Create overlay plot
uhv_color = (90/255, 165/255, 225/255)
line_width = 2

plot_uhv = uhv_df.hvplot(
    x='z', y='I',
    label='UHV cleaved',
    color=uhv_color,
    line_width=line_width
)

plot_alkane = alkane_df.hvplot(
    x='z', y='I',
    label='Contaminated',
    line_width=line_width
)

plot_gold = gold_df.hvplot(
    x='z', y='I',
    label='Gold',
    line_width=line_width
)

plot_spectra = (plot_uhv * plot_alkane * plot_gold).opts(
    title='I-Z Spectroscopy Comparison',
    xlabel='tip - sample distance (Å)',
    ylabel='normalized current (I/I_t)',
    xlim=(-0.1, 5.1),
    width=550,
    height=400,
    fontsize={'title': 12, 'labels': 11, 'xticks': 10, 'yticks': 10, 'legend': 10},
    legend_position='right',
    show_grid=True
)

plot_spectra

:Overlay
   .Curve.UHV_cleaved  :Curve   [z]   (I)
   .Curve.Contaminated :Curve   [z]   (I)
   .Curve.Gold         :Curve   [z]   (I)